In [1]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

student = "student27"
bucket = "spark-q4-2026"
base_path = f"s3a://{bucket}/task1/datasets"

print("student:", student)
print("base_path:", base_path)
print("host:", socket.gethostname())


student: student27
base_path: s3a://spark-q4-2026/task1/datasets
host: vm-en-2


In [2]:
spark = SparkSession.builder \
    .appName(f"{student}_hw1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

spark.sql("SELECT 1 AS ok").show()
print("Spark UI:", spark.sparkContext.uiWebUrl)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/03 15:11:43 WARN Utils: Your hostname, vm-en-2, resolves to a loopback address: 127.0.1.1; using 10.0.0.10 instead (on interface enp2s0)
26/08/03 15:11:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/08/03 15:11:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/03 15:11:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/03 15:11:44 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/08/03 15:11:44 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/08/03 15:11:44 WARN Utils: Service

+---+
| ok|
+---+
|  1|
+---+

Spark UI: http://10.0.0.10:4050


In [3]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS nessie.{student}")

visits_raw = spark.read.parquet(f"{base_path}/VisitsStream")
train_raw = spark.read.parquet(f"{base_path}/trainSearchStream")
search_info_raw = spark.read.parquet(f"{base_path}/SearchInfo")

print("VisitsStream schema")
visits_raw.printSchema()

print("trainSearchStream schema")
train_raw.printSchema()

print("SearchInfo schema")
search_info_raw.printSchema()


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

VisitsStream schema
root
 |-- UserID: integer (nullable = true)
 |-- IPID: integer (nullable = true)
 |-- AdID: integer (nullable = true)
 |-- ViewDate: timestamp (nullable = true)

trainSearchStream schema
root
 |-- SearchID: integer (nullable = true)
 |-- AdID: integer (nullable = true)
 |-- Position: integer (nullable = true)
 |-- ObjectType: integer (nullable = true)
 |-- HistCTR: double (nullable = true)
 |-- IsClick: integer (nullable = true)

SearchInfo schema
root
 |-- SearchID: integer (nullable = true)
 |-- SearchDate: timestamp (nullable = true)
 |-- IPID: integer (nullable = true)
 |-- UserID: integer (nullable = true)
 |-- IsUserLoggedOn: integer (nullable = true)
 |-- SearchQuery: string (nullable = true)
 |-- LocationID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- SearchParams: string (nullable = true)



In [4]:
print("VisitsStream sample")
visits_raw.show(5, truncate=False)

print("trainSearchStream sample")
train_raw.show(5, truncate=False)

print("SearchInfo sample")
search_info_raw.show(5, truncate=False)


VisitsStream sample


+-------+-------+--------+-------------------+
|UserID |IPID   |AdID    |ViewDate           |
+-------+-------+--------+-------------------+
|661479 |1178642|34328519|2015-04-29 03:26:59|
|585938 |1261700|3006003 |2015-04-29 00:07:29|
|3928356|329794 |23804069|2015-04-29 02:46:37|
|726493 |809207 |13811784|2015-04-29 00:52:03|
|2709799|883837 |20737949|2015-04-28 23:04:02|
+-------+-------+--------+-------------------+
only showing top 5 rows
trainSearchStream sample


+---------+--------+--------+----------+--------+-------+
|SearchID |AdID    |Position|ObjectType|HistCTR |IsClick|
+---------+--------+--------+----------+--------+-------+
|107803359|1141788 |2       |1         |NULL    |NULL   |
|107571339|30413707|6       |1         |NULL    |NULL   |
|107483975|27620202|7       |3         |7.03E-4 |0      |
|107402212|19684935|1       |3         |0.005347|0      |
|107805035|28782721|6       |1         |NULL    |NULL   |
+---------+--------+--------+----------+--------+-------+
only showing top 5 rows
SearchInfo sample


[Stage 6:>                                                          (0 + 1) / 1]

+--------+-------------------+-------+-------+--------------+-----------+----------+----------+--------------------------------------------------------------------+
|SearchID|SearchDate         |IPID   |UserID |IsUserLoggedOn|SearchQuery|LocationID|CategoryID|SearchParams                                                        |
+--------+-------------------+-------+-------+--------------+-----------+----------+----------+--------------------------------------------------------------------+
|4198147 |2015-04-29 19:30:53|1837112|1800676|1             |NULL       |2612      |55        |NULL                                                                |
|4278672 |2015-05-12 23:03:41|1737538|3474500|1             |NULL       |1110      |60        |{115:'110-116 см (4-6 лет)', 179:'Платья и юбки', 178:'Для девочек'}|
|4138729 |2015-05-18 07:52:07|1685791|470965 |0             |NULL       |346       |41        |{143:'Fly'}                                                         |
|4236300 |

In [5]:
def pick_col(df, candidates):
    existing = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in existing:
            return existing[candidate.lower()]
    raise ValueError(
        f"None of columns {candidates} found. Existing columns: {df.columns}"
    )


def save_iceberg(df, table_name):
    full_name = f"nessie.{student}.{table_name}"
    print(f"Saving {full_name}")
    df.writeTo(full_name).using("iceberg").createOrReplace()
    print(f"Saved {full_name}; preview from Iceberg table:")
    spark.table(full_name).show(20, truncate=False)


visits_user_col = pick_col(visits_raw, ["UserID", "UserId", "user_id", "userid", "user"])
visits_time_col = pick_col(visits_raw, ["ViewDate", "VisitDate", "EventTime", "Timestamp", "ts", "event_ts"])

train_search_id_col = pick_col(train_raw, ["SearchID", "SearchId", "search_id", "searchid"])
train_click_col = pick_col(train_raw, ["IsClick", "is_click", "click", "clicked"])

search_info_search_id_col = pick_col(search_info_raw, ["SearchID", "SearchId", "search_id", "searchid"])
search_query_col = pick_col(search_info_raw, ["SearchQuery", "Query", "query", "search_query"])

print("Visits user column:", visits_user_col)
print("Visits timestamp column:", visits_time_col)
print("Train SearchID column:", train_search_id_col)
print("Train click column:", train_click_col)
print("SearchInfo SearchID column:", search_info_search_id_col)
print("Search query column:", search_query_col)


Visits user column: UserID
Visits timestamp column: ViewDate
Train SearchID column: SearchID
Train click column: IsClick
SearchInfo SearchID column: SearchID
Search query column: SearchQuery


## Задание 1. Топ-10 самых длинных сессий

Идея решения:

1. Оставляем только id пользователя и время визита.
2. Для каждого пользователя сортируем события по времени.
3. Через оконную функцию `lag` получаем время предыдущего визита.
4. Если предыдущего визита нет или разница больше 15 минут, начинаем новую сессию.
5. Накопительной суммой строим `session_id`.
6. Для каждой сессии считаем длительность: `max(timestamp) - min(timestamp)`.
7. Для каждого пользователя берем самую длинную его сессию.
8. Сортируем по убыванию длительности и берем топ-10.

Оконные функции здесь подходят лучше всего, потому что граница сессии зависит от соседнего события внутри истории конкретного пользователя.


In [6]:
visits = (
    visits_raw
    .select(
        F.col(visits_user_col).cast("long").alias("user_id"),
        F.col(visits_time_col).cast("timestamp").alias("event_ts"),
    )
    .where(F.col("user_id").isNotNull() & F.col("event_ts").isNotNull())
)

user_events_window = Window.partitionBy("user_id").orderBy("event_ts")
user_events_cumulative_window = user_events_window.rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow,
)

visits_with_sessions = (
    visits
    .withColumn("previous_event_ts", F.lag("event_ts").over(user_events_window))
    .withColumn(
        "is_new_session",
        F.when(
            F.col("previous_event_ts").isNull()
            | ((F.col("event_ts").cast("long") - F.col("previous_event_ts").cast("long")) > 15 * 60),
            F.lit(1),
        ).otherwise(F.lit(0)),
    )
    .withColumn("session_id", F.sum("is_new_session").over(user_events_cumulative_window))
)

sessions = (
    visits_with_sessions
    .groupBy("user_id", "session_id")
    .agg(
        F.min("event_ts").alias("session_start"),
        F.max("event_ts").alias("session_end"),
    )
    .withColumn(
        "session_duration_sec",
        F.col("session_end").cast("long") - F.col("session_start").cast("long"),
    )
)

task1_result = (
    sessions
    .groupBy("user_id")
    .agg(F.max("session_duration_sec").alias("max_session_duration_sec"))
    .orderBy(F.desc("max_session_duration_sec"), F.asc("user_id"))
    .limit(10)
)

task1_result.show(10, truncate=False)


[Stage 9:=======================================================> (49 + 1) / 50]

+-------+------------------------+
|user_id|max_session_duration_sec|
+-------+------------------------+
|4143904|358181                  |
|1113291|329463                  |
|1203081|170447                  |
|4263912|80605                   |
|1987990|70544                   |
|563902 |68359                   |
|2853587|53371                   |
|2735474|49881                   |
|1370797|49791                   |
|2271348|48099                   |
+-------+------------------------+



In [7]:
save_iceberg(task1_result, "hw1_task1")


Saving nessie.student27.hw1_task1


Saved nessie.student27.hw1_task1; preview from Iceberg table:
+-------+------------------------+
|user_id|max_session_duration_sec|
+-------+------------------------+
|4143904|358181                  |
|1113291|329463                  |
|1203081|170447                  |
|4263912|80605                   |
|1987990|70544                   |
|563902 |68359                   |
|2853587|53371                   |
|2735474|49881                   |
|1370797|49791                   |
|2271348|48099                   |
+-------+------------------------+



## Задание 2. Топ-10 слов в кликнутых запросах

Идея решения:

1. В `trainSearchStream` оставляем только записи, где был клик по рекламе.
2. Берем `SearchID` таких запросов и делаем `distinct`.
3. Соединяем эти `SearchID` с `SearchInfo`, где лежит текст поискового запроса.
4. Приводим текст запроса к нижнему регистру.
5. Разбиваем строку на слова по пробельным символам.
6. Через `explode` превращаем массив слов в отдельные строки.
7. Считаем частоту каждого слова и берем топ-10.

Другую обработку текста не делаем, потому что в задании сказано не удалять предлоги, короткие слова и прочее.


In [8]:
clicked_searches = (
    train_raw
    .select(
        F.col(train_search_id_col).alias("search_id"),
        F.col(train_click_col).cast("int").alias("is_click"),
    )
    .where(F.col("is_click") == 1)
    .select("search_id")
    .distinct()
)

search_info = (
    search_info_raw
    .select(
        F.col(search_info_search_id_col).alias("search_id"),
        F.col(search_query_col).cast("string").alias("search_query"),
    )
    .where(F.col("search_query").isNotNull())
)

clicked_queries = clicked_searches.join(search_info, on="search_id", how="inner")

task2_result = (
    clicked_queries
    .select(F.explode(F.split(F.lower(F.col("search_query")), r"\s+")).alias("word"))
    .where(F.col("word") != "")
    .groupBy("word")
    .agg(F.count("*").alias("word_count"))
    .orderBy(F.desc("word_count"), F.asc("word"))
    .limit(10)
)

task2_result.show(10, truncate=False)


[Stage 19:=================================================>        (6 + 1) / 7]

+----------+----------+
|word      |word_count|
+----------+----------+
|велосипед |12267     |
|бу        |11059     |
|iphone    |7314      |
|для       |5940      |
|диван     |5566      |
|на        |4857      |
|платье    |4379      |
|коляска   |3759      |
|велосипеды|3720      |
|sony      |3366      |
+----------+----------+



In [9]:
save_iceberg(task2_result, "hw1_task2")


Saving nessie.student27.hw1_task2


Saved nessie.student27.hw1_task2; preview from Iceberg table:
+----------+----------+
|word      |word_count|
+----------+----------+
|велосипед |12267     |
|бу        |11059     |
|iphone    |7314      |
|для       |5940      |
|диван     |5566      |
|на        |4857      |
|платье    |4379      |
|коляска   |3759      |
|велосипеды|3720      |
|sony      |3366      |
+----------+----------+



## Задание 3. Пользователи, заходившие каждый день

Идея решения:

1. Из timestamp визита получаем календарный день через `to_date`.
2. Считаем общее количество разных дней в данных.
3. Для каждого пользователя считаем количество разных дней, когда он заходил на сайт.
4. Оставляем пользователей, у которых это число равно общему числу дней.
5. Ответ задания - одно число: количество таких пользователей.

Датафрейм `every_day_users` сохраняем отдельно, потому что он нужен в задании 4.


In [10]:
visits_days = (
    visits
    .withColumn("day", F.to_date("event_ts"))
    .select("user_id", "day")
    .distinct()
    .cache()
)

total_days = visits_days.select("day").distinct().count()
print("Total measurement days:", total_days)

every_day_users = (
    visits_days
    .groupBy("user_id")
    .agg(F.countDistinct("day").alias("active_days"))
    .where(F.col("active_days") == total_days)
    .select("user_id")
    .cache()
)

task3_result = every_day_users.agg(F.count("*").alias("users_count"))

task3_result.show(truncate=False)


Total measurement days: 26


[Stage 50:=================================================>    (184 + 2) / 200]

+-----------+
|users_count|
+-----------+
|807        |
+-----------+



In [11]:
save_iceberg(task3_result, "hw1_task3")


Saving nessie.student27.hw1_task3
Saved nessie.student27.hw1_task3; preview from Iceberg table:
+-----------+
|users_count|
+-----------+
|807        |
+-----------+



## Задание 4. Уникальные пользователи по дням без ежедневных пользователей

Идея решения:

1. Берем пары `user_id` - `day`, подготовленные в задании 3.
2. Исключаем пользователей, которые заходили каждый день.
3. Для каждого дня считаем количество уникальных пользователей.
4. Сортируем результат по убыванию количества пользователей.

Для исключения используется `left_anti join`: он оставляет только те строки из левой таблицы, для которых не нашлось совпадения справа.


In [12]:
task4_result = (
    visits_days
    .join(every_day_users, on="user_id", how="left_anti")
    .groupBy("day")
    .agg(F.countDistinct("user_id").alias("unique_users"))
    .orderBy(F.desc("unique_users"), F.asc("day"))
)

task4_result.show(100, truncate=False)


[Stage 70:=====================================================>(197 + 1) / 200]

+----------+------------+
|day       |unique_users|
+----------+------------+
|2015-05-12|973248      |
|2015-05-13|847446      |
|2015-05-14|745904      |
|2015-05-06|733027      |
|2015-05-05|731624      |
|2015-05-07|729708      |
|2015-05-11|706240      |
|2015-04-27|676433      |
|2015-05-08|664982      |
|2015-04-28|655791      |
|2015-04-29|641148      |
|2015-05-15|633132      |
|2015-05-04|607205      |
|2015-04-30|596224      |
|2015-05-10|562151      |
|2015-04-26|540051      |
|2015-05-03|511687      |
|2015-04-25|498069      |
|2015-05-16|493577      |
|2015-05-18|488250      |
|2015-05-17|480968      |
|2015-05-02|468010      |
|2015-05-01|452800      |
|2015-05-09|450135      |
|2015-05-19|367777      |
|2015-05-20|184972      |
+----------+------------+



In [13]:
save_iceberg(task4_result, "hw1_task4")


Saving nessie.student27.hw1_task4


Saved nessie.student27.hw1_task4; preview from Iceberg table:
+----------+------------+
|day       |unique_users|
+----------+------------+
|2015-05-12|973248      |
|2015-05-13|847446      |
|2015-05-14|745904      |
|2015-05-06|733027      |
|2015-05-05|731624      |
|2015-05-07|729708      |
|2015-05-11|706240      |
|2015-04-27|676433      |
|2015-05-08|664982      |
|2015-04-28|655791      |
|2015-04-29|641148      |
|2015-05-15|633132      |
|2015-05-04|607205      |
|2015-04-30|596224      |
|2015-05-10|562151      |
|2015-04-26|540051      |
|2015-05-03|511687      |
|2015-04-25|498069      |
|2015-05-16|493577      |
|2015-05-18|488250      |
+----------+------------+
only showing top 20 rows


## Финальная проверка перед сдачей


In [14]:
spark.sql(f"SHOW TABLES IN nessie.{student}").show(100, truncate=False)

for table_name in ["hw1_task1", "hw1_task2", "hw1_task3", "hw1_task4"]:
    full_name = f"nessie.{student}.{table_name}"
    print(full_name)
    spark.table(full_name).show(20, truncate=False)

print("Spark UI:", spark.sparkContext.uiWebUrl)


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|student27|hw1_task1|false      |
|student27|hw1_task2|false      |
|student27|hw1_task3|false      |
|student27|hw1_task4|false      |
+---------+---------+-----------+

nessie.student27.hw1_task1
+-------+------------------------+
|user_id|max_session_duration_sec|
+-------+------------------------+
|4143904|358181                  |
|1113291|329463                  |
|1203081|170447                  |
|4263912|80605                   |
|1987990|70544                   |
|563902 |68359                   |
|2853587|53371                   |
|2735474|49881                   |
|1370797|49791                   |
|2271348|48099                   |
+-------+------------------------+

nessie.student27.hw1_task2
+----------+----------+
|word      |word_count|
+----------+----------+
|велосипед |12267     |
|бу        |11059     |
|iphone    |7314      |
|для       |5940      |
|диван     |55

## ссылки на Spark UI и пути в Iceberg 

Spark UI: http://10.0.0.10:4050

Iceberg таблицы:

nessie.student27.hw1_task1

nessie.student27.hw1_task2

nessie.student27.hw1_task3

nessie.student27.hw1_task4
